# Agent Types

From the [About Workflows](../../../docs/source/workflows/about.md) documentation:

> *"There are several different agents that use language models. They are systems that use LLMs to reason and determine the actions to take and inputs to use for those actions."*

Since agents are functions themselves, they can be used as tools for other agents, enabling hierarchical workflows.

## Supported Agents

NeMo Agent Toolkit provides the following agents ([documentation](../../../docs/source/workflows/about.md)):

| Agent | Documentation | Description |
|-------|---------------|-------------|
| **ReAct Agent** | [ReAct](../../../docs/source/workflows/react-agent/index.md) | Reasoning and acting with explicit thought process |
| **Tool Calling Agent** | [Tool Calling](../../../docs/source/workflows/tool-calling-agent/index.md) | Uses LLM's native function calling capabilities |
| **ReWOO Agent** | [ReWOO](../../../docs/source/workflows/rewoo-agent/index.md) | Planning-first approach for complex multi-step tasks |
| **Reasoning Agent** | [Reasoning](../../../docs/source/workflows/reasoning-agent/index.md) | Enhanced reasoning with thinking capabilities |
| **Router Agent** | [Router](../../../docs/source/workflows/router-agent/index.md) | Routes requests to appropriate sub-agents |
| **Sequential Executor** | [Sequential](../../../docs/source/workflows/sequential-executor/index.md) | Executes functions in sequence |
| **Responses API Agent** | [Responses API](../../../docs/source/workflows/responses-api-and-agent/index.md) | OpenAI Responses API compatible agent |

## Agent Comparison

| Agent | Best For | Pros | Cons |
|-------|----------|------|------|
| **ReAct** | General tasks, debugging | Transparent reasoning, flexible | More tokens used |
| **Tool Calling** | Simple tasks, low latency | Fast, efficient | Less explainable |
| **ReWOO** | Complex multi-step tasks | Better planning | Higher initial latency |

This tutorial covers the three most commonly used agents: ReAct, Tool Calling, and ReWOO.


In [ ]:
import getpass
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - examples will fail")


✅ Environment configured


## Setup: Common Components

First, let's create the LLM and tools we'll use for all agent examples:


In [2]:
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_agent import NatAgent
from nat.utils.sdk.nat_function import NatFunction
from nat.utils.sdk.nat_function_group import NatFunctionGroup
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

tools: list[NatFunction | NatFunctionGroup | NatAgent] = []
try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
    print("✅ LLM and tools created (with calculator)")
except ImportError:
    tools = [time_tool]
    print("✅ LLM and tools created (time only)")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM and tools created (with calculator)


## 1. ReAct Agent

The ReAct (Reasoning + Acting) agent explicitly shows its thought process before taking actions. This makes it excellent for debugging and understanding agent behavior.

### Key Features
- Explicit "Thought" steps before actions
- Clear reasoning chain
- Good for complex reasoning tasks
- More verbose output


In [3]:
from nat.agent.react_agent.register import NatReActAgent

# Create ReAct Agent
react_agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,                           # Show reasoning
    parse_agent_response_max_retries=3,     # Retry on parse errors
    additional_instructions="Think step by step before answering.",
)

react_workflow = NatWorkflow(entrypoint=react_agent)
print("✅ ReAct Agent created")


✅ ReAct Agent created


In [4]:
# Save ReAct config
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

react_config = config_dir / "react_agent.yaml"
react_workflow.save_to_config_file(react_config)
print(f"📄 Saved to: {react_config}")


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


📄 Saved to: configs/react_agent.yaml


## 2. Tool Calling Agent

The Tool Calling agent uses the LLM's native function calling capabilities. It's faster and more efficient but provides less insight into the reasoning process.

### Key Features
- Native LLM function calling
- Lower latency
- Works well with models trained for tool use
- Less verbose output


In [5]:
from nat.agent.tool_calling_agent.register import ToolCallingAgent

# Create Tool Calling Agent
tool_calling_agent = ToolCallingAgent(
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=10,  # Maximum tool calls before stopping
)

tool_calling_workflow = NatWorkflow(entrypoint=tool_calling_agent)
print("✅ Tool Calling Agent created")


✅ Tool Calling Agent created


In [6]:
# Save Tool Calling config
tool_calling_config = config_dir / "tool_calling_agent.yaml"
tool_calling_workflow.save_to_config_file(tool_calling_config)
print(f"📄 Saved to: {tool_calling_config}")


📄 Saved to: configs/tool_calling_agent.yaml


## 3. ReWOO Agent

The ReWOO (Reasoning WithOut Observation) agent creates a plan upfront before executing any tools. This is useful for complex multi-step tasks where planning ahead is beneficial.

### Key Features
- Plans all steps before execution
- Better for complex multi-step tasks
- Reduces back-and-forth with LLM
- Higher initial latency, potentially faster overall


In [7]:
from nat.agent.rewoo_agent.register import ReWOOAgentWorkflow

# Create ReWOO Agent
rewoo_agent = ReWOOAgentWorkflow(
    tools=tools,
    llm=llm,
    verbose=True,
)

rewoo_workflow = NatWorkflow(entrypoint=rewoo_agent)
print("✅ ReWOO Agent created")


✅ ReWOO Agent created


In [8]:
# Save ReWOO config
rewoo_config = config_dir / "rewoo_agent.yaml"
rewoo_workflow.save_to_config_file(rewoo_config)
print(f"📄 Saved to: {rewoo_config}")


📄 Saved to: configs/rewoo_agent.yaml


## Running the Agents

### Via CLI

```bash
# ReAct Agent
nat run --config_file configs/react_agent.yaml --input "What is 25 * 4?"

# Tool Calling Agent  
nat run --config_file configs/tool_calling_agent.yaml --input "What is 25 * 4?"

# ReWOO Agent
nat run --config_file configs/rewoo_agent.yaml --input "What is 25 * 4?"
```

### Via Python


In [12]:
# Test react agent
result = await react_workflow.prompt("What is 25 * 4?")
print(result)


100.0


In [13]:
# Test tool calling agent
result = await tool_calling_workflow.prompt("What is 25 * 4?")
print(result)

The result of 25 * 4 is 100.0.


In [14]:
# Test rewoo agent
result = await rewoo_workflow.prompt("What is 25 * 4?")
print(result)


100


## Choosing the Right Agent

### Use ReAct When:
- You need to debug agent behavior
- Tasks require complex reasoning
- Transparency is important
- You want to see the agent's thought process

### Use Tool Calling When:
- Speed is critical
- Tasks are straightforward
- Using models optimized for function calling
- You don't need detailed reasoning logs

### Use ReWOO When:
- Tasks have multiple dependent steps
- Planning ahead would reduce errors
- You want to minimize LLM round-trips
- Complex workflows with clear sub-tasks

## Summary

✅ **ReAct** - Explicit reasoning, great for debugging  
✅ **Tool Calling** - Fast and efficient for simple tasks  
✅ **ReWOO** - Planning-first for complex multi-step tasks  

## Next Steps

- **[04_functions_and_tools.ipynb](./04_functions_and_tools.ipynb)** - Create custom functions
- **[05_llms.ipynb](./05_llms.ipynb)** - Configure different LLM providers
